# Predict on New Data with TaskPredictor

This notebook demonstrates how to use `TaskPredictor` to predict on new, unseen data after training an Octopus study.

**TaskPredictor vs TaskPredictorTest:**

| | TaskPredictor | TaskPredictorTest |
|---|---|---|
| **Purpose** | Predict on new, unseen data | Analyse study results on held-out test folds |
| **Data** | You provide data explicitly | Uses stored train/test splits from the study |
| **Prediction** | Ensemble average across all outer-split models | Each model predicts only on its own test fold |
| **Deployment** | Can be saved/loaded standalone (no study directory needed) | Requires the full study directory |
| **Use case** | Production inference, scoring new patients/samples | Post-hoc model evaluation, internal validation |

## 1. Train a Study

We split the breast cancer dataset 80/20 — train the study on 80% and hold out 20% as "new" data for prediction.

In [ ]:
import os
import tempfile

import numpy as np
from sklearn.model_selection import train_test_split

from octopus.example_data import load_breast_cancer_data
from octopus.modules import Octo
from octopus.study import OctoClassification
from octopus.types import ModelName

In [ ]:
# Load the breast cancer dataset
df, features, targets = load_breast_cancer_data()

print(f"Full dataset: {df.shape[0]} samples, {len(features)} features")
print(f"Classes: {targets}")
print(f"Target distribution: {df['target'].value_counts().sort_index().to_dict()}")

In [ ]:
# Split 80/20: train the study on train_df, predict on new_df later
train_df, new_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])
train_df = train_df.reset_index(drop=True)
new_df = new_df.reset_index(drop=True)

print(f"Training set: {train_df.shape[0]} samples")
print(f"New data (held-out): {new_df.shape[0]} samples")

In [ ]:
# Create and fit a classification study (2 outer folds, 5 trials for speed)
study = OctoClassification(
    name="predict_example",
    path=os.environ.get("STUDIES_PATH", "./studies"),
    target_metric="AUCROC",
    feature_cols=features,
    target_col="target",
    sample_id_col="index",
    stratification_col="target",
    n_folds_outer=2,
    num_cpus=1,
    workflow=[
        Octo(
            description="step1_octo",
            task_id=0,
            depends_on=None,
            models=[ModelName.ExtraTreesClassifier],
            n_trials=5,
            n_folds_inner=3,
        ),
    ],
)

study.fit(data=train_df)
print(f"Study saved to: {study.output_path}")

## 2. Load the Study

In [ ]:
from octopus.predict.study_io import load_study

study_info = load_study(study.output_path)

print(f"ML type: {study_info.config['ml_type']}")
print(f"Target metric: {study_info.config['target_metric']}")
print(f"Outer splits: {len(study_info.outersplit_dirs)}")
print(f"Features: {len(study_info.config['feature_cols'])}")

## 3. Create a TaskPredictor

In [ ]:
from octopus.predict import TaskPredictor

tp = TaskPredictor(study=study_info, task_id=0)
print(f"Loaded models for {len(tp._models)} outer splits")

## 4. Predict on New Data

Each outer-split model predicts independently, then results are averaged into an ensemble prediction.

In [ ]:
# Ensemble-averaged predictions as a numpy array
predictions = tp.predict(new_df)
print(f"Predictions shape: {predictions.shape}")
print(f"First 10 predictions: {predictions[:10]}")

In [ ]:
# Detailed DataFrame with per-split and ensemble predictions
predictions_df = tp.predict(new_df, df=True)
predictions_df.head(10)

## 5. Predict Probabilities

For classification tasks, `predict_proba` returns class probabilities.

In [ ]:
# Ensemble-averaged probabilities as a numpy array
proba = tp.predict_proba(new_df)
print(f"Probabilities shape: {proba.shape}")
print(f"First 5 rows:\n{proba[:5]}")

In [ ]:
# Detailed DataFrame with per-split and ensemble probabilities
proba_df = tp.predict_proba(new_df, df=True)
proba_df.head(10)

## 6. Evaluate Performance

Score each outer-split model on the held-out new data.

In [ ]:
perf = tp.performance(new_df, metrics=["AUCROC", "ACCBAL"])
perf

## 7. Save and Load for Deployment

Save the predictor to a standalone directory (models + metadata only, no data). Load it back and verify predictions match.

In [ ]:
# Save to a temporary directory
save_dir = tempfile.mkdtemp(prefix="octopus_predictor_")
tp.save(save_dir)
print(f"Saved predictor to: {save_dir}")
print(f"Contents: {os.listdir(save_dir)}")

In [ ]:
# Load the saved predictor (no study directory needed)
tp_loaded = TaskPredictor.load(save_dir)

# Verify predictions match
predictions_loaded = tp_loaded.predict(new_df)
np.testing.assert_array_equal(predictions, predictions_loaded)
print("Predictions match after save/load.")